In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import Perceptron

columns = ["Variance", "Skewness", "Curtosis", "Entropy", "Class"]
df = pd.read_csv("data_banknote_authentication.txt", header=None, names=columns)
display(df.head())
print(df.shape)
print(df.isnull().sum())
display(df.describe())

In [ ]:
features = columns[:-1]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, feature in zip(axes.ravel(), features):
    ax.hist(df[feature], bins=30, edgecolor="black")
    ax.set_title(feature)
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")
fig.suptitle("Feature Histograms")
fig.tight_layout()
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="viridis")
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Correlation Heatmap")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for cls in [0, 1]:
    part = df[df["Class"] == cls]
    ax.scatter(part["Variance"], part["Skewness"], s=18, alpha=0.7, label=f"Class {cls}")
ax.set_xlabel("Variance")
ax.set_ylabel("Skewness")
ax.set_title("Variance vs Skewness by Class")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([df[f].to_numpy() for f in features], tick_labels=features)
ax.set_ylabel("Original feature value")
ax.set_title("Feature Boxplots")
plt.xticks(rotation=20)
plt.show()

In [ ]:
class ScratchPerceptron:
    def __init__(self, learning_rate=0.01, max_epochs=100):
        self.learning_rate = learning_rate
        self.max_epochs = max_epochs

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.weights = np.zeros(X.shape[1], dtype=float)
        self.bias = 0.0
        self.error_history = []
        self.weight_history = []
        self.bias_history = []
        for _ in range(self.max_epochs):
            errors = 0
            for xi, yi in zip(X, y):
                pred = 1 if np.dot(self.weights, xi) + self.bias >= 0 else 0
                update = self.learning_rate * (yi - pred)
                if update != 0:
                    errors += 1
                self.weights += update * xi
                self.bias += update
            self.error_history.append(errors)
            self.weight_history.append(self.weights.copy())
            self.bias_history.append(self.bias)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return (X @ self.weights + self.bias >= 0).astype(int)

X = df[features].to_numpy(dtype=float)
y = df["Class"].to_numpy(dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = ScratchPerceptron(learning_rate=0.01, max_epochs=100).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred)
}
display(pd.DataFrame([metrics]))

In [ ]:
epochs = np.arange(1, len(model.error_history) + 1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, model.error_history, marker="o", markersize=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Misclassified training samples")
ax.set_title("Training Error vs Epoch")
ax.grid(alpha=0.25)
plt.show()

In [ ]:
weight_history = np.asarray(model.weight_history)
fig, ax = plt.subplots(figsize=(8, 5))
for i, feature in enumerate(features):
    ax.plot(epochs, weight_history[:, i], label=feature)
ax.set_xlabel("Epoch")
ax.set_ylabel("Weight value")
ax.set_title("Weight Evolution")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, model.bias_history, marker="o", markersize=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Bias")
ax.set_title("Bias Evolution")
ax.grid(alpha=0.25)
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="viridis")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted 0", "Predicted 1"])
ax.set_yticklabels(["Actual 0", "Actual 1"])
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title("Confusion Matrix")
fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
learning_rate_models = {}
learning_rate_rows = []
fig, ax = plt.subplots(figsize=(8, 5))
for lr in [0.001, 0.01, 0.1]:
    m = ScratchPerceptron(learning_rate=lr, max_epochs=100).fit(X_train_scaled, y_train)
    p = m.predict(X_test_scaled)
    learning_rate_models[lr] = m
    learning_rate_rows.append({
        "Learning Rate": lr,
        "Final Errors": m.error_history[-1],
        "Accuracy": accuracy_score(y_test, p),
        "Precision": precision_score(y_test, p),
        "Recall": recall_score(y_test, p),
        "F1": f1_score(y_test, p)
    })
    ax.plot(np.arange(1, 101), m.error_history, label=f"learning rate = {lr}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Misclassified training samples")
ax.set_title("Learning Rate Comparison")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
display(pd.DataFrame(learning_rate_rows))

In [ ]:
raw_model = ScratchPerceptron(learning_rate=0.01, max_epochs=100).fit(X_train, y_train)
raw_pred = raw_model.predict(X_test)
standard_model = ScratchPerceptron(learning_rate=0.01, max_epochs=100).fit(X_train_scaled, y_train)
standard_pred = standard_model.predict(X_test_scaled)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, 101), raw_model.error_history, label="Unnormalized")
ax.plot(np.arange(1, 101), standard_model.error_history, label="Standardized")
ax.set_xlabel("Epoch")
ax.set_ylabel("Misclassified training samples")
ax.set_title("Effect of Feature Normalization on Convergence")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
normalization_table = pd.DataFrame([
    {
        "Data": "Raw",
        "Errors": raw_model.error_history[-1],
        "Accuracy": accuracy_score(y_test, raw_pred),
        "Precision": precision_score(y_test, raw_pred),
        "Recall": recall_score(y_test, raw_pred),
        "F1": f1_score(y_test, raw_pred)
    },
    {
        "Data": "Standardized",
        "Errors": standard_model.error_history[-1],
        "Accuracy": accuracy_score(y_test, standard_pred),
        "Precision": precision_score(y_test, standard_pred),
        "Recall": recall_score(y_test, standard_pred),
        "F1": f1_score(y_test, standard_pred)
    }
])
display(normalization_table)

In [ ]:
sk_model = Perceptron(max_iter=100, tol=None, eta0=0.01, learning_rate="constant", fit_intercept=True, shuffle=False, random_state=42)
sk_model.fit(X_train_scaled, y_train)
sk_pred = sk_model.predict(X_test_scaled)
comparison = pd.DataFrame([
    {
        "Model": "Scratch",
        "Accuracy": accuracy_score(y_test, standard_pred),
        "Precision": precision_score(y_test, standard_pred),
        "Recall": recall_score(y_test, standard_pred),
        "F1": f1_score(y_test, standard_pred)
    },
    {
        "Model": "Scikit-learn",
        "Accuracy": accuracy_score(y_test, sk_pred),
        "Precision": precision_score(y_test, sk_pred),
        "Recall": recall_score(y_test, sk_pred),
        "F1": f1_score(y_test, sk_pred)
    }
])
display(comparison)
training_summary = pd.DataFrame({
    "Item": ["Dataset size", "Train/Test split", "Learning rate", "Maximum epochs", "Epochs run", "Final training errors", "Final weights", "Final bias", "Accuracy", "Precision", "Recall", "F1-score"],
    "Value": [len(df), f"{len(X_train)} / {len(X_test)}", 0.01, 100, 100, standard_model.error_history[-1], standard_model.weights, standard_model.bias, accuracy_score(y_test, standard_pred), precision_score(y_test, standard_pred), recall_score(y_test, standard_pred), f1_score(y_test, standard_pred)]
})
display(training_summary)
epoch_table = pd.DataFrame({
    "Epoch": np.arange(1, 101),
    "Errors": standard_model.error_history,
    "Bias": standard_model.bias_history
})
for i, feature in enumerate(features):
    epoch_table[f"Weight {i + 1} ({feature})"] = np.asarray(standard_model.weight_history)[:, i]
display(epoch_table)

In [ ]:
X2 = df[["Variance", "Skewness"]].to_numpy(dtype=float)
y2 = y.copy()
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.20, random_state=42)
scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_all_scaled = scaler2.transform(X2)
model2 = ScratchPerceptron(learning_rate=0.01, max_epochs=100).fit(X2_train_scaled, y2_train)
x_min, x_max = X2_all_scaled[:, 0].min() - 0.5, X2_all_scaled[:, 0].max() + 0.5
y_min, y_max = X2_all_scaled[:, 1].min() - 0.5, X2_all_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 350), np.linspace(y_min, y_max, 350))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model2.predict(grid).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(xx, yy, zz, alpha=0.25, levels=[-0.5, 0.5, 1.5])
for cls in [0, 1]:
    pts = X2_all_scaled[y2 == cls]
    ax.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.7, label=f"Class {cls}")
ax.set_xlabel("Standardized Variance")
ax.set_ylabel("Standardized Skewness")
ax.set_title("Two-Feature Perceptron Decision Boundary")
ax.legend()
plt.show()

In [ ]:
z = np.linspace(-8, 8, 600)
step_values = (z >= 0).astype(float)
sigmoid_values = 1 / (1 + np.exp(-z))
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(z, step_values, label="Step")
ax.plot(z, sigmoid_values, label="Sigmoid")
ax.set_xlabel("z")
ax.set_ylabel("Activation output")
ax.set_title("Step and Sigmoid Activation Functions")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
logic_X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

def logic_predict(X, w, b):
    return (np.asarray(X) @ w + b >= 0).astype(int)

def logic_epoch_states(y, learning_rate=0.2, max_epochs=20):
    w = np.zeros(2, dtype=float)
    b = 0.0
    states = [(0, w.copy(), b, int(np.sum(logic_predict(logic_X, w, b) != y)))]
    for epoch in range(1, max_epochs + 1):
        updates = 0
        for xi, yi in zip(logic_X, y):
            pred = 1 if np.dot(w, xi) + b >= 0 else 0
            change = learning_rate * (yi - pred)
            if change != 0:
                updates += 1
            w += change * xi
            b += change
        states.append((epoch, w.copy(), b, updates))
        if updates == 0:
            break
    return states

def logic_update_states(y, learning_rate=0.2, updates=12):
    w = np.zeros(2, dtype=float)
    b = 0.0
    states = []
    count = 0
    while count < updates:
        for xi, yi in zip(logic_X, y):
            pred = 1 if np.dot(w, xi) + b >= 0 else 0
            change = learning_rate * (yi - pred)
            w += change * xi
            b += change
            count += 1
            states.append((count, w.copy(), b, int(np.sum(logic_predict(logic_X, w, b) != y))))
            if count >= updates:
                break
    return states

def draw_logic_boundary(ax, y, w, b, title):
    xx, yy = np.meshgrid(np.linspace(-0.25, 1.25, 250), np.linspace(-0.25, 1.25, 250))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = logic_predict(grid, w, b).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.22, levels=[-0.5, 0.5, 1.5])
    for cls in [0, 1]:
        pts = logic_X[np.asarray(y) == cls]
        ax.scatter(pts[:, 0], pts[:, 1], s=70, edgecolors="black", linewidths=0.5)
        for px, py in pts:
            ax.text(px + 0.03, py + 0.03, str(cls), fontsize=8)
    ax.set_xlim(-0.25, 1.25)
    ax.set_ylim(-0.25, 1.25)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(title)
    ax.grid(alpha=0.2)

In [ ]:
and_y = np.array([0, 0, 0, 1])
and_states = logic_epoch_states(and_y)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax in axes.ravel():
    ax.axis("off")
for ax, (epoch, w, b, updates) in zip(axes.ravel(), and_states):
    ax.axis("on")
    title = "Initialization" if epoch == 0 else f"Epoch {epoch} ({updates} update errors)"
    draw_logic_boundary(ax, and_y, w, b, title)
fig.suptitle("AND Gate - Boundary from Initialization to Convergence")
fig.tight_layout()
plt.show()

In [ ]:
or_y = np.array([0, 1, 1, 1])
or_states = logic_epoch_states(or_y)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax in axes.ravel():
    ax.axis("off")
for ax, (epoch, w, b, updates) in zip(axes.ravel(), or_states):
    ax.axis("on")
    title = "Initialization" if epoch == 0 else f"Epoch {epoch} ({updates} update errors)"
    draw_logic_boundary(ax, or_y, w, b, title)
fig.suptitle("OR Gate - Boundary from Initialization to Convergence")
fig.tight_layout()
plt.show()

In [ ]:
xor_y = np.array([0, 1, 1, 0])
xor_states = logic_update_states(xor_y, updates=12)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, (update, w, b, errors) in zip(axes.ravel(), xor_states):
    draw_logic_boundary(ax, xor_y, w, b, f"Update {update} - {errors} misclassified")
fig.suptitle("XOR Gate - Consecutive Perceptron Updates")
fig.tight_layout()
plt.show()

In [ ]:
xor_epoch_states = logic_epoch_states(xor_y, max_epochs=20)
xor_epoch_errors = [state[3] for state in xor_epoch_states[1:]]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.arange(1, len(xor_epoch_errors) + 1), xor_epoch_errors, marker="o")
ax.set_xlabel("Epoch")
ax.set_ylabel("Update errors")
ax.set_title("XOR Gate - Perceptron Does Not Reach Zero Errors")
ax.set_xticks(np.arange(1, len(xor_epoch_errors) + 1))
ax.grid(alpha=0.25)
plt.show()